In [1]:
import sys
sys.path.append("../")  # Goes up to "project/"
import test_operators as top
import test_dataloader as tdl
import test_loss as tl
import test_latticeConv as tge 
import parameters as var
var.print_parameters()

*********** Configuration parameters ***********
* β=2, Nx=64, Nt=64
* Variables=8192
* m0=-0.1884
* blocks_x=2, blocks_t=2 (for the aggregation)
* SAP vectors for the loss function Nv=30
* Fake test vectors generated Nv=10
* Number of confs=500
* Confs used for training=50
* Device: cuda:0
* Precision: double
* DOF on fine grid: 8192
* DOF on coarse grid: 80
* Gauge equivariant model: True
************************************************


# Testing operators

In [2]:
nv, nx, nt, blocks_x, blocks_t = 2, 4, 4, 2, 2
opsTest = top.TestOperators(nv,nx,nt,blocks_x,blocks_t)
opsTest.assembleP()
opsTest.assemblePdagg()
# In case I want to print the operators
#opsTest.printP()
#opsTest.printPdagg()

Nv=2, Nx=4, Nt=4, Blocks X=2, Blocks T=2
Test vectors not orthonormalized


In [3]:
#Checking P*^T = Pdagg
opsTest.testP_Pdagg()
#Checking that local orthonormalization of test vectors is correct.
opsTest.testOrthonormalization()

P*^T = Pdagg, i.e. all good.
Test vectors are locally orthonormalized


# Testing dataloader

In [9]:
dloaderTest = tdl.TestDataloader(5,"other")
dloaderTest.testDimensions()
dloaderTest.testDataType()
dloaderTest.testConfZero()
print()
dloaderTest.testTVector()
print()
dloaderTest.testPlaquette()
print()
dloaderTest.testPlaquettePt()
print()
dloaderTest.testPlaquetteDataLoader()
print("*************\nCheck H5FD\n*************")
dloaderTest = tdl.TestDataloader(5,"h5")
dloaderTest.testDimensions()
dloaderTest.testDataType()
dloaderTest.testConfZero()
print()
dloaderTest.testTVector()
print()
dloaderTest.testPlaquette()
print()
dloaderTest.testPlaquettePt()
print()
dloaderTest.testPlaquetteDataLoader()

gconfs and tvectors dimensions correct
dtype of gconfs torch.float64
dtype of tvectors torch.complex128
Check that conf 0 is correctly read. Compare with the actual numbers from the conf file.
- - - Uμ(t,x) - - -
U0(0,0)= (0.2556388959218151+0.9667723387084902j) torch.complex128 cuda:0
U1(0,0)= (-0.48252952056273996+0.875879707371674j) torch.complex128 cuda:0
U0(1,0)= (0.14394714652617133+0.989585377320716j) torch.complex128 cuda:0
U1(1,0)= (0.671609917853468-0.7409049319857974j) torch.complex128 cuda:0
U1(15,10)= (0.7178042334009688-0.6962449874230061j) torch.complex128 cuda:0
All elements of the gauge conf are U(1) variables

Check that test vector 0 from conf 0 coincides with the info in the file
- - - Phi_μ(t,x) - - -
Phi_0(0,0)= (-0.1684731884379952+0.09385017207658897j) torch.complex128 cuda:0
Phi_1(0,0)= (0.16017270343191894+0.04861070248862853j) torch.complex128 cuda:0
Phi_0(1,0)= (0.12304118576055745+0.06855103186017578j) torch.complex128 cuda:0
Phi_1(1,0)= (-0.049835013519419

# Testing loss function

In [7]:
batch_size = 100
lossTest = tl.TestLoss(batch_size)
lossTest.testTrivialCase()
lossTest.testRandomCase()
lossTest.testRemainderTV()

Trivial case succesful
Loss when interpolator is assembled with SAP vectors and evaluated on random vectors: tensor(5306.0526, device='cuda:0', dtype=torch.float64)
Loss when interpolator is assembled with random vectors and evaluated on SAP vectors: tensor(667.0439, device='cuda:0', dtype=torch.float64)


AssertionError: NV has to be smaller than 30, otherwise there are not enough test vectors for this test

In [2]:
#import convert_to_pt
#convert_to_pt.binaryPlaq2ptPlaq()

In [3]:
import sys
sys.path.append("../")  # Goes up to "project/"
import test_latticeConv as tge 
import parameters as var
import numpy as np
import torch
import torch.nn as nn

In [2]:
#Check parallel transport
lcnn = tge.TestLCNN(100)
lcnn.check_u1_vars()
print(lcnn.w.shape)
wxkm = lcnn.check_parallel_transport()
w = lcnn.w
lcnn.print_parallel_transport()

Number of local transforming real variables 2
U shape torch.Size([100, 2, 32, 32])
Type torch.complex128
W shape torch.Size([100, 1, 32, 32])
Type torch.complex128
Variables are all elements of U(1)
torch.Size([100, 1, 32, 32])
W(-1,0)=(0.8294010423136913-0.5586536592638922j), Wkxmu(0,0)=(0.8294010423136913-0.5586536592638922j)
W(-1,0)=(0.8294010423136913-0.5586536592638922j), Wkxmu(0,0)=(0.8294010423136913-0.5586536592638922j)
W(-1,0)=(0.8294010423136913-0.5586536592638922j), Wkxmu(0,0)=(0.8294010423136913-0.5586536592638922j)
W(-1,0)=(0.8294010423136913-0.5586536592638922j), Wkxmu(0,0)=(0.8294010423136913-0.5586536592638922j)

W(0,-1)=(-0.9579144021127303+0.28705399879639465j), Wkxmu(0,0)=(-0.9579144021127303+0.28705399879639465j)
W(0,-1)=(-0.9579144021127303+0.28705399879639465j), Wkxmu(0,0)=(-0.9579144021127303+0.28705399879639465j)
W(0,-1)=(-0.9579144021127303+0.28705399879639465j), Wkxmu(0,0)=(-0.9579144021127303+0.28705399879639465j)
W(0,-1)=(-0.9579144021127303+0.28705399879639

In [3]:
u, w = lcnn.LConv_test()
w[0,0,0,0].item()

Gauge equivariant convolution, in_features=1, out_features=4, ksize=3
Input shape torch.Size([100, 1, 32, 32])
Output shape torch.Size([100, 4, 32, 32])


(0.9004493334778911+4.071788701891942j)

In [4]:
tge.test_LCNN_layers()

Input shape torch.Size([100, 1, 32, 32])
W shape after convolution torch.Size([100, 20, 32, 32])
U shape after convolution torch.Size([100, 2, 32, 32])


In [9]:
u = torch.Tensor(100,3,3)

def transporter(u,orientation,mu,k):
    p_transporter = 1
    u_mu = u[:,mu] #(B,NT,NX)
    #x+mu
    if orientation == -1:
        #We are just multiplying U(1) numbers, so the order of operations is not relevant. For a different group this 
        #function would have to be rewritten respecting the order. 
        for i in range(k):
            p_transporter *= torch.roll(u_mu,shifts=i*orientation,dims=1+mu)
    #x-mu
    elif orientation == 1:
        for i in range(1,k+1):
            p_transporter *= torch.roll(u_mu.conj(),shifts=i*orientation,dims=1+mu)
    return p_transporter.unsqueeze(1)    

In [22]:
u = torch.zeros([100,2,3,3],dtype=var.PREC_COMPLEX)
u[0,0,0,0] = 1j
u[0,0,0,1] = 2j
u[0,0,0,2] = 3j
u[0,0,1,0] = 4j
u[0,0,1,1] = 5j
u[0,0,1,2] = 6j
u[0,0,2,0] = 7j
u[0,0,2,1] = 8j
u[0,0,2,2] = 9j

In [32]:
orientation = 1
mu = 0
k = 2
#product = u*torch.roll(u[:,mu],shifts=orientation,dims=1+mu).unsqueeze(1)
pt = transporter(u,orientation,mu,k)
print("Transporter function",pt[0,0])
print("Product",product[0,0])

Transporter function tensor([[-28.-0.j, -40.-0.j, -54.-0.j],
        [ -7.-0.j, -16.-0.j, -27.-0.j],
        [ -4.-0.j, -10.-0.j, -18.-0.j]], dtype=torch.complex128)
Product tensor([[ -7.+0.j, -16.+0.j, -27.+0.j],
        [ -4.+0.j, -10.+0.j, -18.+0.j],
        [-28.+0.j, -40.+0.j, -54.+0.j]], dtype=torch.complex128)


# Testing parallel transport along arbitrary paths

In [1]:
import numpy as np
import torch
import torch.nn as nn

In [20]:
def Tp(u,path):
    #path = [-1,-2,-1,+2,+2] 1 = \hat{t}, 2 = \hat{x}, I don't use +0 -0 for obvious reasons
    #u.shape = (Batch,2,NT,NX)
    p_transporter = 1
    for p in path:
        mu  = abs(p)-1
        if p > 0:
            #Hp = U^+_p(x-p)
            #x-p
            #u[:,0] = torch.roll(u[:,0],shifts=1,dims=1+mu)
            #u[:,1] = torch.roll(u[:,1],shifts=1,dims=1+mu)
            u = torch.roll(u, shifts=1, dims=2+mu)
            p_transporter *= u[:,mu].conj()
        elif p < 0:
            #x+p
            p_transporter *= u[:,mu] #We roll both components ...
            #u[:,0] = torch.roll(u[:,0],shifts=-1,dims=1+mu)
            #u[:,1] = torch.roll(u[:,1],shifts=-1,dims=1+mu)
            u = torch.roll(u, shifts=-1, dims=2+mu) 
        else:
            raise("p should be positive or negative, not zero")
    return p_transporter
        

In [21]:
path = [-1,-2,-1,2,2]
B, Nx, Nt = 100, 4, 4
# Random phases in [0, 2π)
theta = 2 * torch.pi * torch.rand(B, 2, Nt, Nx, dtype=torch.float64)
# U(1) variables with complex128 precision
u = torch.exp(1j * theta)
u_roll1 = torch.zeros(B, 2, Nt, Nx, dtype=torch.complex128)
u_roll2 = torch.zeros(B, 2, Nt, Nx, dtype=torch.complex128)
print(u.shape)   # torch.Size([B, 2, Nt, Nx])
print(u.dtype)   # torch.complex128

torch.Size([100, 2, 4, 4])
torch.complex128


In [22]:
mu = 0
u_roll1[:,0] = torch.roll(u[:,0],shifts=1,dims=1+mu)
u_roll1[:,1] = torch.roll(u[:,1],shifts=1,dims=1+mu)
u_roll2 = torch.roll(u, shifts=1, dims=2+mu)

In [23]:
print("Original tensor")
print(u[0,0,1,0])
print(u[0,1,1,0])
print("Roll1 tensor")
print(u_roll1[0,0,0,0])
print(u_roll1[0,1,0,0])
print("Roll2 tensor")
print(u_roll2[0,0,0,0])
print(u_roll2[0,1,0,0])
print("Comparing rolled tensors")
torch.allclose(u_roll1,u_roll2)

Original tensor
tensor(0.9360+0.3520j, dtype=torch.complex128)
tensor(0.8006+0.5992j, dtype=torch.complex128)
Roll1 tensor
tensor(-0.8352-0.5499j, dtype=torch.complex128)
tensor(-0.6660+0.7459j, dtype=torch.complex128)
Roll2 tensor
tensor(-0.8352-0.5499j, dtype=torch.complex128)
tensor(-0.6660+0.7459j, dtype=torch.complex128)
Comparing rolled tensors


True

In [37]:
parallel_transport = Tp(u,path)
print("Parallel transport shape",parallel_transport.shape)
#Constructing the transport along [-1,-2,-1,2,2] manually.
nx, nt = 3, 2
b = 0
pt = u[b,0,nt,nx] * u[b,1,(nt+1)%Nt,nx] * u[b,0,(nt+1)%Nt,(nx+1)%Nx] * u.conj()[b,1,(nt+1+1)%Nt,(nx+1-1)%Nx] * u.conj()[b,1,(nt+1+1)%Nt,(nx+1-1-1)%Nx]
print("Manual transport",pt)
print("Using the function",parallel_transport[0])

Parallel transport shape torch.Size([100, 4, 4])
Manual transport tensor(-0.5243+0.8516j, dtype=torch.complex128)
Using the function tensor([[ 0.4871+0.8733j, -0.4881+0.8728j, -0.4934+0.8698j,  0.2656-0.9641j],
        [-0.4713+0.8820j,  0.0068+1.0000j,  0.3371-0.9415j,  0.8632+0.5049j],
        [ 0.0698-0.9976j,  0.9772-0.2125j, -0.7154-0.6987j, -0.5243+0.8516j],
        [-0.2623-0.9650j,  0.9627-0.2705j, -0.9906-0.1364j,  0.4193+0.9078j]],
       dtype=torch.complex128)
